# 01 — Data preparation

This notebook reproduces the data-cleaning and weather-aggregation procedure described in Sections 3.1–3.3 of the paper. It starts from the two original hourly files, creates a single spatially aggregated weather record per timestamp, merges energy and weather information, removes non-informative variables, imputes the small number of missing predictor values, audits outliers, and exports the cleaned modelling dataset.

### Inputs

- `data/raw/energy_dataset.csv`
- `data/raw/weather_features.csv`

Both files originate from the publicly available Kaggle dataset *Hourly energy demand, generation and weather*.

### Outputs

- `data/processed/energy_dataset_clean.csv`
- `outputs/audits/data_cleaning_summary.csv`
- `outputs/audits/outlier_summary.csv`

No observations from the target variable are imputed, and statistically identified outliers are retained because they may represent genuine electricity-market conditions.

## 1. Configuration and imports

In [3]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
from IPython.display import display

TIMEZONE = "Europe/Paris"  # CET/CEST convention used in the manuscript
EXPECTED_CITIES = ["Barcelona", "Bilbao", "Madrid", "Seville", "Valencia"]
WEATHER_SOURCE_COLUMNS = ["temp", "pressure", "humidity", "wind_speed"]
WEATHER_OUTPUT_NAMES = {
    "temp": "temperature",
    "pressure": "pressure",
    "humidity": "humidity",
    "wind_speed": "wind_speed",
}
TARGET_SOURCE = "price day ahead"
TARGET_OUTPUT = "price"
EXCLUDED_ENERGY_COLUMNS = [
    "forecast solar day ahead",
    "forecast wind onshore day ahead",
    "total load forecast",
    "price actual",
]

# This cell works when the notebook is launched either from the repository root
# or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR = PROJECT_ROOT / "outputs" / "audits"

ENERGY_FILE = RAW_DIR / "energy_dataset.csv"
WEATHER_FILE = RAW_DIR / "weather_features.csv"
OUTPUT_FILE = PROCESSED_DIR / "energy_dataset_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"Project root: {PROJECT_ROOT}")

Python: 3.12.2
pandas: 2.2.2
Project root: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather


## 2. Helper functions

Timestamps are parsed as UTC and then converted to the CET/CEST timezone used in the study. This preserves the two distinct occurrences of the repeated hour at the autumn daylight-saving transition.

In [5]:
def read_timestamped_csv(path: Path, time_column: str) -> pd.DataFrame:
    '''Read a CSV and convert its timestamp column to the study timezone.'''
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}\n"
            "Place the original Kaggle files in data/raw/."
        )

    frame = pd.read_csv(path)
    if time_column not in frame.columns:
        raise KeyError(f"{path.name} does not contain '{time_column}'.")

    parsed = pd.to_datetime(frame[time_column], utc=True, errors="raise")
    frame[time_column] = parsed.dt.tz_convert(TIMEZONE)
    return frame


def iqr_outlier_summary(frame: pd.DataFrame) -> pd.DataFrame:
    '''Return IQR-based diagnostics without removing any observations.'''
    rows = []
    for column in frame.select_dtypes(include=np.number).columns:
        values = frame[column].dropna()
        if values.nunique() < 2:
            count = 0
        else:
            q1, q3 = values.quantile([0.25, 0.75])
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            count = int(((values < lower) | (values > upper)).sum())
        rows.append({
            "variable": column,
            "outlier_count": count,
            "outlier_percent": 100.0 * count / len(values),
        })
    return pd.DataFrame(rows).sort_values(
        "outlier_percent", ascending=False, ignore_index=True
    )

## 3. Load the original datasets

In [7]:
energy_raw = read_timestamped_csv(ENERGY_FILE, "time")
weather_raw = read_timestamped_csv(WEATHER_FILE, "dt_iso")

print("Energy dataset:", energy_raw.shape)
print("Weather dataset:", weather_raw.shape)
print("Energy period:", energy_raw["time"].min(), "to", energy_raw["time"].max())
print("Weather period:", weather_raw["dt_iso"].min(), "to", weather_raw["dt_iso"].max())

Energy dataset: (35064, 29)
Weather dataset: (178396, 17)
Energy period: 2015-01-01 00:00:00+01:00 to 2018-12-31 23:00:00+01:00
Weather period: 2015-01-01 00:00:00+01:00 to 2018-12-31 23:00:00+01:00


## 4. Prepare and spatially aggregate the weather data

The original weather file contains observations for five Spanish cities and may contain more than one record for the same city and timestamp. Exact duplicate rows are removed first. If several non-identical records remain for a city–timestamp pair, the first record in source order is retained, reproducing the preparation rule used for the study.

Temperature, pressure, humidity and wind speed are then averaged across Barcelona, Bilbao, Madrid, Seville and Valencia. The result contains one national-level meteorological record per timestamp.

In [9]:
required_weather = {"dt_iso", "city_name", *WEATHER_SOURCE_COLUMNS}
missing_weather_columns = required_weather.difference(weather_raw.columns)
if missing_weather_columns:
    raise KeyError(f"Missing weather columns: {sorted(missing_weather_columns)}")

weather = weather_raw.loc[:, ["dt_iso", "city_name", *WEATHER_SOURCE_COLUMNS]].copy()
weather = weather.rename(columns={"dt_iso": "time"})

observed_cities = sorted(weather["city_name"].str.strip().dropna().unique().tolist())
if observed_cities != sorted(EXPECTED_CITIES):
    raise ValueError(
        f"Unexpected cities. Expected {sorted(EXPECTED_CITIES)}, "
        f"but found {observed_cities}."
    )

exact_duplicates = int(weather.duplicated().sum())
weather = weather.drop_duplicates()

duplicate_city_hours = int(weather.duplicated(["time", "city_name"]).sum())
weather = weather.drop_duplicates(["time", "city_name"], keep="first")

city_coverage = weather.groupby("time")["city_name"].nunique()
if not city_coverage.eq(len(EXPECTED_CITIES)).all():
    incomplete = city_coverage[city_coverage.ne(len(EXPECTED_CITIES))]
    raise ValueError(
        f"{len(incomplete)} timestamps do not contain all five cities. "
        "Inspect the source weather data before continuing."
    )

weather_hourly = (
    weather.groupby("time", as_index=False)[WEATHER_SOURCE_COLUMNS]
    .mean()
    .rename(columns=WEATHER_OUTPUT_NAMES)
    .sort_values("time")
    .reset_index(drop=True)
)

assert weather_hourly["time"].is_unique
print("Exact duplicate weather rows removed:", exact_duplicates)
print("Additional duplicate city–timestamp rows removed:", duplicate_city_hours)
print("Aggregated weather shape:", weather_hourly.shape)
display(weather_hourly.head())

Exact duplicate weather rows removed: 3076
Additional duplicate city–timestamp rows removed: 0
Aggregated weather shape: (35064, 5)


,time,temperature,pressure,humidity,wind_speed
0,2015-01-01 00:00:00+01:00,272.491463,1016.4,82.4,2.0
1,2015-01-01 01:00:00+01:00,272.512700,1016.2,82.4,2.0
2,2015-01-01 02:00:00+01:00,272.099137,1016.8,82.0,2.4
3,2015-01-01 03:00:00+01:00,272.089469,1016.6,82.0,2.4
4,2015-01-01 04:00:00+01:00,272.145900,1016.6,82.0,2.4


## 5. Prepare the energy data and merge

The target is renamed from `price_day_ahead` to `price` for consistency with the modelling notebooks. This is a naming change only. Realised price and the original load, solar and onshore-wind forecast columns are excluded because they are not part of the information set used by the final models.

In [11]:
if TARGET_SOURCE not in energy_raw.columns:
    raise KeyError(f"Energy data do not contain target '{TARGET_SOURCE}'.")

energy = energy_raw.copy()
exact_energy_duplicates = int(energy.duplicated().sum())
energy = energy.drop_duplicates()

if energy["time"].duplicated().any():
    raise ValueError("The energy dataset contains multiple non-identical rows per timestamp.")

energy = energy.rename(columns={TARGET_SOURCE: TARGET_OUTPUT})

excluded_energy_columns = [
    column for column in EXCLUDED_ENERGY_COLUMNS if column in energy.columns
]
energy = energy.drop(columns=excluded_energy_columns)

merged = energy.merge(
    weather_hourly,
    on="time",
    how="inner",
    validate="one_to_one",
)

if len(merged) != len(energy) or len(merged) != len(weather_hourly):
    raise ValueError(
        "The merge did not preserve all hourly observations: "
        f"energy={len(energy)}, weather={len(weather_hourly)}, merged={len(merged)}."
    )

merged = merged.sort_values("time").reset_index(drop=True)
print("Exact duplicate energy rows removed:", exact_energy_duplicates)
print("Excluded energy columns:", excluded_energy_columns)
print("Merged shape:", merged.shape)

Exact duplicate energy rows removed: 0
Excluded energy columns: ['forecast solar day ahead', 'forecast wind onshore day ahead', 'total load forecast', 'price actual']
Merged shape: (35064, 29)


## 6. Validate the hourly timeline

Continuity is checked in UTC so that daylight-saving transitions do not create false gaps or duplicates in local clock time.

In [13]:
if not merged["time"].is_monotonic_increasing:
    raise ValueError("Timestamps are not sorted.")
if not merged["time"].is_unique:
    raise ValueError("Timestamps are not unique.")

utc_time = merged["time"].dt.tz_convert("UTC")
hourly_steps = utc_time.diff().dropna()
non_hourly_steps = hourly_steps.ne(pd.Timedelta(hours=1))
if non_hourly_steps.any():
    raise ValueError(f"Detected {int(non_hourly_steps.sum())} non-hourly time steps.")

print("Observations:", len(merged))
print("First timestamp:", merged["time"].iloc[0])
print("Last timestamp:", merged["time"].iloc[-1])
print("Hourly UTC continuity: verified")

Observations: 35064
First timestamp: 2015-01-01 00:00:00+01:00
Last timestamp: 2018-12-31 23:00:00+01:00
Hourly UTC continuity: verified


## 7. Remove constant and entirely missing variables

A variable is non-informative when it has no observed values or only one distinct observed value. Missing values are excluded when counting distinct values; otherwise a column containing zeros and `NaN` values would incorrectly appear to have two values.

In [15]:
EXPECTED_NONINFORMATIVE = {
    "generation fossil coal-derived gas",
    "generation fossil oil shale",
    "generation fossil peat",
    "generation geothermal",
    "generation hydro pumped storage aggregated",
    "generation marine",
    "generation wind offshore",
    "forecast wind offshore eday ahead",
}

candidate_columns = [column for column in merged.columns if column != "time"]
noninformative = {
    column for column in candidate_columns
    if merged[column].nunique(dropna=True) <= 1
}

if noninformative != EXPECTED_NONINFORMATIVE:
    raise ValueError(
        "The detected non-informative variables differ from those used in the study.\n"
        f"Expected: {sorted(EXPECTED_NONINFORMATIVE)}\n"
        f"Detected: {sorted(noninformative)}"
    )

data = merged.drop(columns=sorted(noninformative)).copy()
print(f"Removed {len(noninformative)} non-informative variables:")
for column in sorted(noninformative):
    print(" -", column)
print("Shape after removal:", data.shape)

Removed 8 non-informative variables:
 - forecast wind offshore eday ahead
 - generation fossil coal-derived gas
 - generation fossil oil shale
 - generation fossil peat
 - generation geothermal
 - generation hydro pumped storage aggregated
 - generation marine
 - generation wind offshore
Shape after removal: (35064, 21)


## 8. Standardise variable names

In [17]:
data.columns

Index(['time', 'generation biomass', 'generation fossil brown coal/lignite',
       'generation fossil gas', 'generation fossil hard coal',
       'generation fossil oil', 'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation nuclear',
       'generation other', 'generation other renewable', 'generation solar',
       'generation waste', 'generation wind onshore', 'total load actual',
       'price', 'temperature', 'pressure', 'humidity', 'wind_speed'],
      dtype='object')

In [18]:
data.columns = (
    data.columns
    .str.lower()
    .str.replace("generation", "gen", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

if data.columns.duplicated().any():
    raise ValueError("Column-name standardisation created duplicate names.")

display(pd.DataFrame({"variable": data.columns}))

,variable
0,time
1,gen_biomass
2,gen_fossil_brown_coal_lignite
3,gen_fossil_gas
4,gen_fossil_hard_coal
5,gen_fossil_oil
6,gen_hydro_pumped_storage_consumption
7,gen_hydro_run-of-river_and_poundage
8,gen_hydro_water_reservoir
9,gen_nuclear


## 9. Missing-value audit and interpolation

The target contains no missing values and is never imputed. Linear time interpolation is applied only to numeric predictor columns. The original dataset contains very little missingness, and no rows are removed.

In [20]:
missing_before = data.isna().sum().rename("missing_before")
missing_table = missing_before[missing_before.gt(0)].to_frame()
missing_table["percent_before"] = (
    100.0 * missing_table["missing_before"] / len(data)
)
display(missing_table)

if data[TARGET_OUTPUT].isna().any():
    raise ValueError("The target contains missing values; target imputation is not permitted.")

data = data.set_index("time")
predictor_columns = [column for column in data.columns if column != TARGET_OUTPUT]
numeric_predictors = data[predictor_columns].select_dtypes(include=np.number).columns

data.loc[:, numeric_predictors] = data[numeric_predictors].interpolate(
    method="time",
    limit_direction="both",
)

remaining_missing = data.isna().sum()
if remaining_missing.any():
    unresolved = remaining_missing[remaining_missing.gt(0)]
    raise ValueError(f"Missing values remain after interpolation:\n{unresolved}")

data = data.reset_index()
print("Missing values after interpolation:", int(data.isna().sum().sum()))

,missing_before,percent_before
gen_biomass,19,0.054187
gen_fossil_brown_coal_lignite,18,0.051335
gen_fossil_gas,18,0.051335
gen_fossil_hard_coal,18,0.051335
gen_fossil_oil,19,0.054187
gen_hydro_pumped_storage_consumption,19,0.054187
gen_hydro_run-of-river_and_poundage,19,0.054187
gen_hydro_water_reservoir,18,0.051335
gen_nuclear,17,0.048483
gen_other,18,0.051335


Missing values after interpolation: 0


## 10. Outlier audit

The 1.5×IQR rule is used only as a descriptive diagnostic. No observations are removed or winsorised because extreme generation, load and price values can represent genuine market behaviour.

In [22]:
outlier_summary = iqr_outlier_summary(data.drop(columns="time"))
display(outlier_summary.head(15).style.format({"outlier_percent": "{:.2f}%"}))

,variable,outlier_count,outlier_percent
0,gen_hydro_pumped_storage_consumption,3763,10.73%
1,pressure,2264,6.46%
2,gen_fossil_gas,2185,6.23%
3,gen_other,1267,3.61%
4,price,841,2.40%
5,wind_speed,833,2.38%
6,gen_wind_onshore,377,1.08%
7,gen_hydro_water_reservoir,341,0.97%
8,gen_waste,328,0.94%
9,gen_fossil_oil,246,0.70%


## 11. Final validation and export

In [24]:
expected_rows = 35_064
if len(data) != expected_rows:
    raise ValueError(f"Expected {expected_rows:,} rows, obtained {len(data):,}.")
if data["time"].duplicated().any():
    raise ValueError("Duplicate timestamps remain in the cleaned dataset.")
if data.isna().any().any():
    raise ValueError("The cleaned dataset still contains missing values.")

cleaning_summary = pd.DataFrame([
    {"item": "energy_rows_loaded", "value": len(energy_raw)},
    {"item": "weather_rows_loaded", "value": len(weather_raw)},
    {"item": "exact_energy_duplicates_removed", "value": exact_energy_duplicates},
    {"item": "exact_weather_duplicates_removed", "value": exact_duplicates},
    {"item": "duplicate_city_hours_removed", "value": duplicate_city_hours},
    {"item": "energy_columns_excluded", "value": len(excluded_energy_columns)},
    {"item": "noninformative_variables_removed", "value": len(noninformative)},
    {"item": "missing_values_before_interpolation", "value": int(missing_before.sum())},
    {"item": "missing_values_after_interpolation", "value": int(data.isna().sum().sum())},
    {"item": "final_rows", "value": len(data)},
    {"item": "final_columns", "value": data.shape[1]},
])

data.to_csv(OUTPUT_FILE, index=False)
cleaning_summary.to_csv(AUDIT_DIR / "data_cleaning_summary.csv", index=False)
outlier_summary.to_csv(AUDIT_DIR / "outlier_summary.csv", index=False)

display(cleaning_summary)
print(f"Cleaned dataset saved to: {OUTPUT_FILE}")
print(f"Audit files saved to: {AUDIT_DIR}")

,item,value
0,energy_rows_loaded,35064
1,weather_rows_loaded,178396
2,exact_energy_duplicates_removed,0
3,exact_weather_duplicates_removed,3076
4,duplicate_city_hours_removed,0
5,energy_columns_excluded,4
6,noninformative_variables_removed,8
7,missing_values_before_interpolation,292
8,missing_values_after_interpolation,0
9,final_rows,35064


Cleaned dataset saved to: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather/data/processed/energy_dataset_clean.csv
Audit files saved to: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather/outputs/audits


## Reproducibility note

The exported file is the common cleaned source used by the subsequent feature-construction and forecasting notebooks. Calendar encodings, historical windows, lagged exogenous predictors and rolling predictors are intentionally constructed later, at each forecast origin, so that their availability constraints can be enforced explicitly.